<a href="https://colab.research.google.com/github/alish-ba15/DL_Assignment/blob/main/Copy_of_DL_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generation 1 — DNN Baseline

In [ ]:
!pip install -q tensorflow==2.15.0 keras==2.15.0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
!unzip /content/archive.zip -d /content/dataset

In [ ]:
df = pd.read_csv('/content/dataset/Dataset.csv')
df = df.sample(20000, random_state=42)

In [ ]:
# Data Preprocessing
numerical_columns = df.select_dtypes(include=np.number).columns
imputer = SimpleImputer(strategy='median')
df[numerical_columns] = imputer.fit_transform(df[numerical_columns])

df['SBP'] = df['SBP'].replace(0, 0.01)
df['ShockIndex'] = df['HR'] / df['SBP']
df['PulsePressure'] = df['SBP'] - df['DBP']
df['LowOxygenFlag'] = (df['O2Sat'] < 90).astype(int)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.fillna(df.median(numeric_only=True))


# Feature Selection and Scaling
X = df.drop(['SepsisLabel','Patient_ID'], axis=1)
y = df['SepsisLabel']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)


# Class Imbalance Handling Using SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_scaled, y = smote.fit_resample(
    X_scaled,
    y
)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:
def create_dnn_model(optimizer):

    model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )

    return model

In [ ]:
# Class Weight Computation
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))


# Early Stopping Configuration
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


# DNN with SGD Optimizer
sgd_model = create_dnn_model(
    SGD(learning_rate=0.001, momentum=0.9)
)

history_sgd = sgd_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

In [ ]:
# DNN with Adam Optimizer
adam_model = create_dnn_model(
    Adam(learning_rate=0.0001, clipnorm=1.0)
)

history_adam = adam_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

# ADAM Evaluation
predictions = adam_model.predict(X_test)
predictions = (predictions > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall:", recall_score(y_test, predictions))
print("F1:", f1_score(y_test, predictions))

In [ ]:
# SGD Evaluation
sgd_predictions = sgd_model.predict(X_test)
sgd_predictions = (sgd_predictions > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, sgd_predictions))
print("Precision:", precision_score(y_test, sgd_predictions))
print("Recall:", recall_score(y_test, sgd_predictions))
print("F1:", f1_score(y_test, sgd_predictions))

In [ ]:
# =====================================================
# SGD vs ADAM LOSS CURVES
# =====================================================

plt.figure(figsize=(10,5))

# SGD Training Loss
plt.plot(
    history_sgd.history['loss'],
    label='SGD Train Loss'
)

# SGD Validation Loss
plt.plot(
    history_sgd.history['val_loss'],
    label='SGD Validation Loss'
)

# Adam Training Loss
plt.plot(
    history_adam.history['loss'],
    label='Adam Train Loss'
)

# Adam Validation Loss
plt.plot(
    history_adam.history['val_loss'],
    label='Adam Validation Loss'
)

plt.xlabel("Epochs")

plt.ylabel("Loss")

plt.title("SGD vs Adam Loss Comparison")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# =====================================================
# SGD vs ADAM ACCURACY CURVES
# =====================================================

plt.figure(figsize=(10,5))

# SGD Accuracy
plt.plot(
    history_sgd.history['accuracy'],
    label='SGD Train Accuracy'
)

plt.plot(
    history_sgd.history['val_accuracy'],
    label='SGD Validation Accuracy'
)

# Adam Accuracy
plt.plot(
    history_adam.history['accuracy'],
    label='Adam Train Accuracy'
)

plt.plot(
    history_adam.history['val_accuracy'],
    label='Adam Validation Accuracy'
)

plt.xlabel("Epochs")

plt.ylabel("Accuracy")

plt.title("SGD vs Adam Accuracy Comparison")

plt.legend()

plt.grid(True)

plt.show()

**SGD Confusion Matrix and Evaluation Matrix**

In [ ]:
cm = confusion_matrix(
    y_test,
    sgd_predictions
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted Label")

plt.ylabel("True Label")

plt.title("Confusion Matrix")

plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        sgd_predictions
    )
)

**Conclusion**

Based on the evaluation metrics and confusion matrix, the DNN model trained with SGD outperformed the Adam optimizer. SGD achieved better predictive performance and showed stronger generalization on unseen data. Therefore, SGD was chosen as the optimal optimizer for the baseline model.